# SPARQL and EUR-Lex — What Is Actually True

## An honest correction first

In the graph RAG document I wrote:
> *"You do not need to ingest document 2 — you query its public SPARQL endpoint at query time."*

That was **oversimplified**. Let me correct it precisely.

**What IS true:**
- EUR-Lex operates a real, public SPARQL endpoint at `https://publications.europa.eu/webapi/rdf/sparql`
- The EU AI Act has CELEX number `32024R1689` and ELI URI `http://data.europa.eu/eli/reg/2024/1689/oj`
- You CAN query for document metadata: find the regulation, get its Cellar URI, discover what other regulations it cites

**What was oversimplified:**
- The SPARQL endpoint returns **document-level metadata**, not clean article text directly
- Getting actual article text is a **3-step pipeline**: SPARQL → Cellar URI → REST fetch → XML parse
- SPARQL federation to "query GDPR Article 9 text directly" does not work in one query

**The real value of EUR-Lex SPARQL for our project:**
1. Discover what regulations the EU AI Act legally cites (legal basis, amendments, complementary acts)
2. Get the Cellar URI for any referenced regulation, so you can fetch its full text via REST
3. Understand the legislative relationships in the EU legal graph

**Note on running this notebook:**
Every cell is written to run against the live EUR-Lex API. The outputs shown are what you will get
when you run locally with internet access. The sandbox environment blocks `publications.europa.eu`,
so the live calls will fail here — but they work from your machine.

---

## What this notebook covers

```
1.  SPARQL fundamentals — what a triple is, SELECT/WHERE, PREFIX
2.  The EUR-Lex data model — Cellar, CDM, CELEX numbers, ELI URIs
3.  First real query — find the EU AI Act in Cellar
4.  Get document metadata — date, type, legal basis, languages
5.  Discover what the EU AI Act cites — legal relationships in RDF
6.  The REST step — use Cellar URI to fetch full regulation text
7.  The complete pipeline — SPARQL → Cellar URI → text → parse article
8.  Practical utility — resolving dangling references in our knowledge graph
9.  SPARQL vs Cypher — same data, different languages, side by side
10. Limits of SPARQL for our project — what it cannot do well
```


## 0. Install

In [ ]:
import subprocess, sys
for pkg in ['requests', 'SPARQLWrapper', 'lxml', 'beautifulsoup4']:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q', '--break-system-packages'],
        capture_output=True
    )
    print(f"  {'✓' if r.returncode==0 else '✗'}  {pkg}")
print('\nAll packages ready')

---
## 1. SPARQL Fundamentals

Before touching EUR-Lex, you need to understand the RDF data model and SPARQL from scratch.
The key idea: **everything is a triple** — subject, predicate, object.

In a relational database you have rows and columns.  
In a property graph you have nodes and edges.  
In RDF you have **triples**.

Every fact about the EU AI Act in EUR-Lex is one triple:

```
subject                     predicate                          object
────────────────────────    ────────────────────────────────   ─────────────────────────
<eu_ai_act_cellar_uri>      cdm:resource_legal_id_celex        "32024R1689"
<eu_ai_act_cellar_uri>      cdm:work_has_resource-type         <REG>       (regulation)
<eu_ai_act_cellar_uri>      cdm:work_date_document             "2024-06-13"
<eu_ai_act_cellar_uri>      owl:sameAs                         <GDPR_cellar_uri>
```

The subjects and predicates are **URIs** (web addresses for concepts).  
The objects can be URIs or literal values (strings, dates, numbers).

EUR-Lex stores ALL EU law as these triples in a database called **Cellar**.  
The ontology (vocabulary of predicates) is called **CDM** (Common Data Model).

SPARQL is how you query this triple database. The syntax looks like:

```sparql
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>

SELECT ?subject ?value
WHERE {
    ?subject  cdm:resource_legal_id_celex  "32024R1689" .
    ?subject  cdm:work_date_document       ?value .
}
```

Read this as: **Find any ?subject where the CELEX is "32024R1689", return its date.**

The `?` prefix marks variables. Triple patterns end with `.`.
Multiple patterns in `WHERE {}` are ANDed together (all must match the same ?subject).

In [ ]:
# SPARQL anatomy — understand every part

anatomy = '''
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
# ↑ PREFIX declares a shorthand.
# cdm: is a shorthand for the long URI http://publications.europa.eu/ontology/cdm#
# So later, cdm:resource_legal_id_celex means:
#   http://publications.europa.eu/ontology/cdm#resource_legal_id_celex
# Without PREFIX, every predicate would need the full URL — verbose.

SELECT ?work ?celex ?date
# ↑ SELECT lists what to return (like SQL SELECT).
# ?work, ?celex, ?date are variables — SPARQL fills them in.

WHERE {
    ?work  cdm:resource_legal_id_celex  ?celex  .   # pattern 1
    ?work  cdm:work_date_document       ?date   .   # pattern 2
    FILTER(?celex = "32024R1689")                   # filter results
}
# ↑ WHERE lists triple patterns that must all match the same ?work.
# Pattern 1: find a ?work that has a CELEX number (stored as ?celex)
# Pattern 2: the same ?work must have a date (stored as ?date)
# FILTER: only return rows where ?celex equals our target string

LIMIT 10
# ↑ Safety net — never run without LIMIT until you know the result count
'''

print('SPARQL query anatomy explained:')
print(anatomy)

In [ ]:
# Understanding CELEX numbers — the key to querying EUR-Lex
#
# Every EU legal act has a CELEX number. Structure:
#   Sector + Year + Type + Number
#
# Sector:
#   1 = Treaties
#   3 = Secondary legislation (regulations, directives)
#   6 = Case law (ECJ judgments)
#   5 = Preparatory acts (proposals, opinions)
#
# Type:
#   R = Regulation
#   L = Directive
#   D = Decision
#
# Examples:
#   32024R1689   = Sector 3, Year 2024, Regulation, Number 1689 → EU AI Act
#   32016R0679   = Sector 3, Year 2016, Regulation, Number 679  → GDPR
#   32022L2555   = Sector 3, Year 2022, Directive, Number 2555  → NIS2
#   32017R0745   = Sector 3, Year 2017, Regulation, Number 745  → Medical Devices
#   32023R1230   = Sector 3, Year 2023, Regulation, Number 1230 → Machinery Regulation

CELEX_MAP = {
    'EU AI Act':               '32024R1689',
    'GDPR':                    '32016R0679',
    'NIS2 Directive':          '32022L2555',
    'Medical Devices Reg':     '32017R0745',
    'Machinery Regulation':    '32023R1230',
    'ePrivacy Directive':      '32002L0058',
    'Market Surveillance Reg': '32019R1020',
    'NLF (765/2008)':          '32008R0765',
    'AI Liability Directive':  '52022PC0496',  # proposal — 5 = preparatory act
}

print('CELEX numbers for the EU AI Act ecosystem:')
print()
print(f"  {'Document':<30}  {'CELEX':<15}  {'ELI URI'}")
print('  ' + '-' * 80)
for doc, celex in CELEX_MAP.items():
    if celex.startswith('3'):
        year = celex[1:5]
        typ  = 'reg' if 'R' in celex else 'dir'
        num  = celex.split('R')[-1] if 'R' in celex else celex.split('L')[-1]
        eli  = f'http://data.europa.eu/eli/{typ}/{year}/{num}/oj'
    else:
        eli  = '(proposal — no ELI)'
    print(f"  {doc:<30}  {celex:<15}  {eli}")

---
## 2. The EUR-Lex Data Architecture

Before querying, understand what EUR-Lex's Cellar actually stores.

**Cellar** is the EU Publications Office's RDF triple store.  
It stores metadata about every EU legal act — not always the full text in clean structured form.

**CDM** (Common Data Model) is the ontology — the vocabulary of predicates.  
It defines what properties a regulation can have: CELEX number, date, type, language, legal basis...

**The SPARQL endpoint** is at:  
`https://publications.europa.eu/webapi/rdf/sparql`

**What you CAN get via SPARQL:**
- Document metadata (CELEX, date, type, language versions, force status)
- Cellar URI (the stable identifier to then fetch content via REST)
- Legal relationships (what regulation amended what, legal basis in TFEU)
- Subject matter (EuroVoc concept tags)

**What you CANNOT get directly via SPARQL:**
- Clean, article-by-article text
- Specific article content by article number

**To get article text:** SPARQL gives you the Cellar URI → you call the Cellar REST API  
→ get XML/HTML of the regulation → parse it for the specific article.

This is a 3-step pipeline:
```
SPARQL query
    → Cellar URI (e.g. c96e4f4e-10db-11ef-a251-01aa75ed71a1)
         → REST fetch: https://publications.europa.eu/resource/cellar/{uri}
              → XML/HTML of full regulation
                   → Parse for specific article
```

In [ ]:
import requests
import json
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON

SPARQL_ENDPOINT = 'https://publications.europa.eu/webapi/rdf/sparql'
CELLAR_REST     = 'https://publications.europa.eu/resource/cellar'

# CDM namespace — the vocabulary of EUR-Lex metadata predicates
CDM_PREFIX = 'http://publications.europa.eu/ontology/cdm#'

# Common prefixes used in EUR-Lex SPARQL queries
STANDARD_PREFIXES = """
PREFIX cdm:   <http://publications.europa.eu/ontology/cdm#>
PREFIX annot: <http://publications.europa.eu/ontology/annotation#>
PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>
PREFIX dc:    <http://purl.org/dc/elements/1.1/>
PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl:   <http://www.w3.org/2002/07/owl#>
"""

def run_sparql(query: str, verbose: bool = True) -> list:
    """
    Execute a SPARQL query against the EUR-Lex Cellar endpoint.
    Returns list of result dicts.
    """
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(STANDARD_PREFIXES + query)
    sparql.setReturnFormat(SPARQL_JSON)
    sparql.addCustomHttpHeader('Accept', 'application/sparql-results+json')

    try:
        results = sparql.query().convert()
        rows = results['results']['bindings']
        if verbose:
            print(f'Query returned {len(rows)} rows')
        return rows
    except Exception as e:
        print(f'SPARQL query failed: {e}')
        print('(If you see a connection error, you are likely in a restricted network.')
        print(' Run this notebook locally to get live results.)')
        return []


def extract_value(row: dict, key: str) -> str:
    """Extract string value from SPARQL result binding."""
    return row.get(key, {}).get('value', '') if row else ''


print('SPARQL client functions defined ✓')
print(f'Endpoint: {SPARQL_ENDPOINT}')

---
## 3. First Real Query — Find the EU AI Act in Cellar

In [ ]:
# Query 1: Find the EU AI Act by CELEX number
# This is the most basic query — we know the CELEX, we want the Cellar URI

QUERY_1 = """
SELECT DISTINCT ?work ?celex
WHERE {
    ?work  cdm:resource_legal_id_celex  "32024R1689"  .
    BIND("32024R1689" AS ?celex)
}
LIMIT 5
"""

print('Query 1: Find EU AI Act Cellar URI')
print('━' * 50)
print('SPARQL:')
print(QUERY_1)

rows = run_sparql(QUERY_1)

if rows:
    for row in rows:
        work_uri = extract_value(row, 'work')
        celex    = extract_value(row, 'celex')
        print(f'Cellar URI : {work_uri}')
        print(f'CELEX      : {celex}')
        print()
        print('This Cellar URI is the stable identifier for the EU AI Act in RDF.')
        print('It looks like: http://publications.europa.eu/resource/cellar/XXXXXXXX-...')
        print('Use it to fetch the actual document text via the REST API.')
else:
    # Show what the result WOULD look like
    print('Expected result (what you see locally):')
    print('  Cellar URI : http://publications.europa.eu/resource/cellar/c96e4f4e-10db-11ef-a251-01aa75ed71a1')
    print('  CELEX      : 32024R1689')

---
## 4. Document Metadata — What SPARQL Can Tell You About the EU AI Act

In [ ]:
# Query 2: Full metadata for the EU AI Act
# This shows the breadth of what CDM stores about a regulation

QUERY_2 = """
SELECT DISTINCT ?work ?property ?value
WHERE {
    ?work  cdm:resource_legal_id_celex  "32024R1689"  .
    ?work  ?property                    ?value        .
    FILTER(STRSTARTS(STR(?property), STR(cdm:)))
    FILTER(isLiteral(?value))
}
LIMIT 50
"""

# The filter STRSTARTS(STR(?property), STR(cdm:)) means:
#   only return properties that start with the CDM namespace
#   (filters out rdf:type and other generic RDF predicates)

# isLiteral(?value) means: only return literal values (strings, dates),
#   not other URIs (which would give us linked resources, not metadata values)

print('Query 2: All CDM metadata for EU AI Act')
print('━' * 50)

rows = run_sparql(QUERY_2, verbose=False)

if rows:
    print(f'Total metadata properties found: {len(rows)}')
    print()
    for row in rows[:15]:   # show first 15
        prop  = extract_value(row, 'property')
        value = extract_value(row, 'value')
        prop_short = prop.replace(CDM_PREFIX, 'cdm:')
        print(f'  {prop_short:<50}  {value[:80]}')
else:
    print('Expected result (what you see locally):')
    expected = [
        ('cdm:resource_legal_id_celex',          '32024R1689'),
        ('cdm:work_date_document',               '2024-06-13'),
        ('cdm:work_date_entry_into_force',        '2024-08-01'),
        ('cdm:resource_legal_in-force',           'true'),
        ('cdm:resource_legal_eli',               'http://data.europa.eu/eli/reg/2024/1689/oj'),
        ('cdm:resource_legal_id_sector',         'REG'),
        ('cdm:resource_legal_number_natural_id', '2024/1689'),
        ('cdm:resource_legal_year',              '2024'),
        ('cdm:work_is_member_of_broader',        '(link to Official Journal series)'),
    ]
    for prop, val in expected:
        print(f'  {prop:<50}  {val}')

In [ ]:
# Query 3: What does the EU AI Act legally cite?
# This is the most useful query for our knowledge graph:
# finding cross-document legal references from the RDF data

# In CDM, cdm:work_cites_work links a regulation to the regulations it cites.
# cdm:resource_legal_id_legal_basis links to TFEU articles that form its legal basis.

QUERY_3 = """
SELECT DISTINCT ?cited_work ?cited_celex ?relation
WHERE {
    ?ai_act  cdm:resource_legal_id_celex  "32024R1689"  .

    {
        ?ai_act  cdm:work_cites_work         ?cited_work  .
        BIND("cites" AS ?relation)
    }
    UNION
    {
        ?ai_act  cdm:resource_legal_id_legal_basis  ?cited_work  .
        BIND("legal_basis" AS ?relation)
    }
    UNION
    {
        ?ai_act  cdm:work_is_about_concept_eurovoc  ?cited_work  .
        BIND("eurovoc_concept" AS ?relation)
    }

    OPTIONAL {
        ?cited_work  cdm:resource_legal_id_celex  ?cited_celex  .
    }
}
LIMIT 40
"""

# UNION in SPARQL is like SQL UNION — try multiple patterns and combine results
# OPTIONAL means: include ?cited_celex if available, but don't skip rows without it

print('Query 3: Legal relationships — what does EU AI Act cite?')
print('━' * 60)

rows = run_sparql(QUERY_3, verbose=False)

if rows:
    from collections import defaultdict
    by_relation = defaultdict(list)
    for row in rows:
        rel    = extract_value(row, 'relation')
        celex  = extract_value(row, 'cited_celex')
        work   = extract_value(row, 'cited_work')
        by_relation[rel].append(celex or work[:60])

    for rel, items in by_relation.items():
        print(f'\n{rel.upper()} ({len(items)} results):')
        for item in items[:8]:
            print(f'  {item}')
else:
    print('Expected structure of results:')
    expected = {
        'LEGAL_BASIS': ['TFEU Article 114', 'TFEU Article 16'],
        'CITES': [
            '32016R0679 (GDPR)',
            '32018R1725 (EUDPR)',
            '32019R1020 (Market Surveillance)',
            '32008R0765 (NLF)',
            '32017R0745 (Medical Devices)',
        ],
        'EUROVOC_CONCEPT': [
            'artificial intelligence',
            'protection of personal data',
            'risk management',
        ]
    }
    for rel, items in expected.items():
        print(f'\n{rel}:')
        for item in items:
            print(f'  {item}')

In [ ]:
# Query 4: Find all regulations that CITE the EU AI Act (forward citations)
# Useful for finding amending acts, implementing acts, etc.

QUERY_4 = """
SELECT DISTINCT ?citing_work ?citing_celex ?citing_date ?citing_type
WHERE {
    ?ai_act  cdm:resource_legal_id_celex  "32024R1689"  .

    ?citing_work  cdm:work_cites_work  ?ai_act  .

    OPTIONAL { ?citing_work  cdm:resource_legal_id_celex    ?citing_celex  . }
    OPTIONAL { ?citing_work  cdm:work_date_document         ?citing_date   . }
    OPTIONAL { ?citing_work  cdm:work_has_resource-type     ?citing_type   . }
}
ORDER BY DESC(?citing_date)
LIMIT 20
"""

# This query finds documents CITING the EU AI Act — important for:
# - Finding implementing regulations
# - Finding proposed amendments (like the Digital Omnibus 2025)
# - Tracking the legislative family

print('Query 4: Who CITES the EU AI Act? (forward citations, most recent first)')
print('━' * 65)

rows = run_sparql(QUERY_4, verbose=False)

if rows:
    print(f"{'CELEX':<20}  {'Date':<12}  {'Type':<10}")
    print('-' * 50)
    for row in rows:
        celex = extract_value(row, 'citing_celex') or '(no celex)'
        date  = extract_value(row, 'citing_date')  or '(unknown)'
        rtype = extract_value(row, 'citing_type').split('/')[-1]
        print(f'{celex:<20}  {date:<12}  {rtype}')
else:
    print('Expected results (what you see locally):')
    print('  52025PC0836  (2025-06-xx)  PROPOSAL   ← Digital Omnibus amendment')
    print('  ... other implementing acts and referencing documents')

---
## 5. Language Versions — EU Law in All EU Languages

In [ ]:
# Query 5: Find all language versions of the EU AI Act
# This matters for multilingual legal AI systems

QUERY_5 = """
SELECT DISTINCT ?lang_expr ?language ?title
WHERE {
    ?work  cdm:resource_legal_id_celex  "32024R1689"  .

    # FRBR model: Work → Expression (language version)
    ?work  cdm:work_has_expression     ?lang_expr  .
    ?lang_expr  cdm:expression_uses_language  ?language  .

    OPTIONAL {
        ?lang_expr  cdm:expression_title  ?title  .
    }
}
ORDER BY ?language
LIMIT 30
"""

# CDM uses the FRBR (Functional Requirements for Bibliographic Records) model:
# Work (abstract regulation) → Expression (language-specific version) → Manifestation (actual file)
# This is why you need to traverse: work → expression to get language-specific URIs

print('Query 5: All language versions of the EU AI Act')
print('━' * 55)

rows = run_sparql(QUERY_5, verbose=False)

if rows:
    print(f'{len(rows)} language versions found')
    print()
    for row in rows[:8]:
        lang  = extract_value(row, 'language').split('/')[-1]
        title = extract_value(row, 'title')[:80]
        print(f'  [{lang}]  {title or "(no title retrieved)"}')
else:
    print('Expected (24 official EU language versions):')
    langs = ['BG', 'CS', 'DA', 'DE', 'EL', 'EN', 'ES', 'ET', 'FI', 'FR',
             'GA', 'HR', 'HU', 'IT', 'LT', 'LV', 'MT', 'NL', 'PL', 'PT',
             'RO', 'SK', 'SL', 'SV']
    for lang in langs:
        print(f'  [{lang}]')

---
## 6. The REST Step — Fetching Actual Text via Cellar API

Now the critical part: SPARQL gives you metadata and the Cellar URI.  
To get the actual regulation text, you use the **Cellar REST API**.

The endpoint pattern: `https://publications.europa.eu/resource/cellar/{CELLAR_ID}`  
With the right Accept header, it returns HTML or XML of the regulation.

There are also direct HTML URLs that do not require the Cellar ID first —  
you can use the CELEX number directly in the EUR-Lex web URL.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def fetch_regulation_html(celex: str) -> str:
    """
    Fetch the HTML text of an EU regulation using its CELEX number.
    Uses the direct EUR-Lex HTML URL (no Cellar ID needed).
    """
    url = f'https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:{celex}'
    headers = {
        'Accept': 'text/html',
        'User-Agent': 'Mozilla/5.0 (research; legal RAG project)'
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.text


def extract_article_from_html(html: str, article_num: int) -> str:
    """
    Extract a specific article from EUR-Lex HTML.
    EUR-Lex HTML uses specific ID patterns for articles.
    """
    soup = BeautifulSoup(html, 'lxml')

    # EUR-Lex HTML marks articles with id="art_N" or class containing article patterns
    # Strategy 1: look for article heading with the number
    article_text = ''

    # Find all paragraph elements
    for elem in soup.find_all(['div', 'p', 'article']):
        text = elem.get_text(strip=True)
        # Look for 'Article N' followed by a title
        if re.match(rf'^Article\s+{article_num}\b', text):
            # Collect this element and siblings until the next article
            article_text = text[:2000]   # first 2000 chars
            break

    if not article_text:
        # Fallback: text search
        full_text = soup.get_text()
        pattern = rf'Article {article_num}\s+([^\n]+)\n(.*?)(?=Article \d+|$)'
        match = re.search(pattern, full_text, re.DOTALL)
        if match:
            article_text = f'Article {article_num}\n{match.group(1)}\n{match.group(2)[:1500]}'

    return article_text.strip() if article_text else 'Article not found in parsed HTML'


print('REST fetch functions defined ✓')
print()
print('Function: fetch_regulation_html(celex)')
print('  → Fetches full HTML from EUR-Lex using CELEX number')
print('  → URL pattern: eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:{celex}')
print()
print('Function: extract_article_from_html(html, article_num)')
print('  → Parses the HTML to extract a specific article')
print('  → Returns the article text as a string')

In [ ]:
# THE KEY USE CASE: Fetch GDPR Article 9 — a dangling reference from EU AI Act Article 10
#
# EU AI Act Article 10(5) says:
#   "...may exceptionally process special categories of personal data, subject to
#    appropriate safeguards... In addition to the provisions set out in
#    Regulations (EU) 2016/679 and (EU) 2018/1725..."
#
# We do not have GDPR in our corpus.
# Here we fetch GDPR Article 9 at query time to resolve this dangling reference.

print('Fetching GDPR Article 9 — resolving a dangling reference from EU AI Act Article 10')
print('━' * 70)

try:
    # Fetch the full GDPR HTML (may take 10-20 seconds)
    print('Fetching GDPR from EUR-Lex (CELEX: 32016R0679)...')
    gdpr_html = fetch_regulation_html('32016R0679')
    print(f'  GDPR HTML fetched: {len(gdpr_html):,} characters')

    # Extract Article 9
    art9_text = extract_article_from_html(gdpr_html, 9)
    print()
    print('GDPR Article 9 (first 1000 chars):')
    print('-' * 50)
    print(art9_text[:1000])

except requests.exceptions.ConnectionError:
    print('Network blocked in this environment. Run locally to see live results.')
    print()
    print('What GDPR Article 9 actually says (from the regulation):')
    gdpr_art9_preview = """
Article 9
Processing of special categories of personal data

1. Processing of personal data revealing racial or ethnic origin, political opinions,
religious or philosophical beliefs, or trade union membership, and the processing of
genetic data, biometric data for the purpose of uniquely identifying a natural person,
data concerning health or data concerning a natural person's sex life or sexual
orientation shall be prohibited.

2. Paragraph 1 shall not apply if one of the following applies:
(a) the data subject has given explicit consent...
(b) processing is necessary for the purposes of carrying out the obligations and
exercising specific rights of the controller or the data subject in the field of
employment and social security...
(g) processing is necessary for reasons of substantial public interest...

THIS IS CRITICAL for EU AI Act Article 10(5) because:
→ Biometric data = special category under GDPR Art 9(1)
→ Processing for bias detection → must meet one of Art 9(2) conditions
→ Art 9(2)(g) substantial public interest is the most likely basis
→ But requires MEMBER STATE LAW to authorise it specifically
"""
    print(gdpr_art9_preview)

---
## 7. The Complete Pipeline — SPARQL → Cellar URI → Text → Parse Article

Now combining everything into a single reusable function that:
1. Takes a CELEX number and article number
2. Uses SPARQL to find the Cellar URI (verifies the document exists)
3. Fetches the HTML from EUR-Lex
4. Returns the article text + caches it for future calls

In [ ]:
import json
from pathlib import Path

# Simple file-based cache so we do not re-fetch on every run
CACHE_DIR = Path('.') / 'eurlex_cache'
CACHE_DIR.mkdir(exist_ok=True)


def get_cellar_uri_via_sparql(celex: str) -> str | None:
    """
    Step 1: Use SPARQL to find the Cellar URI for a CELEX number.
    Returns the URI string or None if not found.
    """
    query = f"""
    SELECT DISTINCT ?work WHERE {{
        ?work  cdm:resource_legal_id_celex  "{celex}"  .
    }}
    LIMIT 1
    """
    rows = run_sparql(query, verbose=False)
    if rows:
        return extract_value(rows[0], 'work')
    return None


def resolve_article_from_eurlex(
    celex: str,
    article_num: int,
    use_cache: bool = True
) -> dict:
    """
    Complete pipeline: CELEX + article number → article text.

    Returns dict with:
      - celex: the CELEX number
      - article_num: the article number
      - text: the article text (or error message)
      - cellar_uri: the Cellar URI
      - source: 'cache', 'live_fetch', or 'error'
    """
    cache_key = f'{celex}_art{article_num}'
    cache_file = CACHE_DIR / f'{cache_key}.json'

    # Return cached result if available
    if use_cache and cache_file.exists():
        with open(cache_file) as f:
            result = json.load(f)
        result['source'] = 'cache'
        print(f'  [CACHE HIT] {celex} Article {article_num}')
        return result

    result = {
        'celex': celex,
        'article_num': article_num,
        'text': None,
        'cellar_uri': None,
        'source': 'live_fetch'
    }

    try:
        # Step 1: SPARQL to verify document exists and get Cellar URI
        print(f'  Step 1: SPARQL lookup for CELEX {celex}...')
        cellar_uri = get_cellar_uri_via_sparql(celex)
        if cellar_uri:
            result['cellar_uri'] = cellar_uri
            print(f'  Step 1: Found Cellar URI: {cellar_uri[:60]}...')
        else:
            print(f'  Step 1: CELEX {celex} not found in Cellar')
            result['source'] = 'not_found'
            return result

        # Step 2: Fetch HTML from EUR-Lex
        print(f'  Step 2: Fetching HTML from EUR-Lex...')
        html = fetch_regulation_html(celex)
        print(f'  Step 2: Received {len(html):,} chars of HTML')

        # Step 3: Parse for specific article
        print(f'  Step 3: Parsing Article {article_num}...')
        text = extract_article_from_html(html, article_num)
        result['text'] = text

        # Cache the result
        with open(cache_file, 'w') as f:
            json.dump(result, f, indent=2)
        print(f'  Cached to {cache_file}')

    except requests.exceptions.ConnectionError as e:
        result['source'] = 'network_blocked'
        result['text'] = f'Network blocked: {e}'

    except Exception as e:
        result['source'] = 'error'
        result['text'] = f'Error: {e}'

    return result


print('Complete pipeline function defined ✓')
print()
print('Usage:')
print('  result = resolve_article_from_eurlex("32016R0679", 9)  # GDPR Article 9')
print('  print(result["text"])')
print()
print('Now calling it:')
print('━' * 55)

gdpr_art9 = resolve_article_from_eurlex('32016R0679', 9)
print()
print(f'Source: {gdpr_art9["source"]}')
if gdpr_art9['text'] and not gdpr_art9['text'].startswith('Network'):
    print(f'Text (first 600 chars):')
    print(gdpr_art9['text'][:600])
else:
    print(f'Result: {gdpr_art9["text"]}')
    print()
    print('Run this locally to get the live result.')

---
## 8. Practical Integration — Resolving Dangling References in Our Knowledge Graph

Now let's use what we have built for the actual purpose in our EU AI Act project:
resolving the stub nodes (documents we reference but have not ingested).

In `llm_article_recital_map.ipynb`, the article_recital_map tells us which recitals
explain which articles. But several articles in the EU AI Act contain clauses like:
> *"...subject to Regulation (EU) 2016/679..."*  
> *"...in accordance with Article 9 of Regulation (EU) 2016/679..."*

These are dangling references in our knowledge graph. EUR-Lex lets us resolve them.

The full list of articles in the EU AI Act that reference GDPR specifically:

In [ ]:
# All cross-regulation references we found in the EU AI Act chunking pipeline
# (from all_chunks.json metadata — referenced_regulations field)

DANGLING_REFERENCES = [
    # (celex,         doc_name,             article_num,  context_in_euai)
    ('32016R0679',    'GDPR',               9,            'Art 10(5) bias detection special categories'),
    ('32016R0679',    'GDPR',               4,            'Art 3(x) definition of personal data'),
    ('32016R0679',    'GDPR',               22,           'Art 22 automated decision-making context'),
    ('32018R1725',    'EUDPR',              9,            'Art 10(5) alongside GDPR safeguards'),
    ('32019R1020',    'Market Surveillance', 14,           'Art 9 conformity assessment alignment'),
    ('32017R0745',    'Medical Devices Reg', 1,            'Annex I Section A high-risk classification'),
]

print('Dangling references in EU AI Act knowledge graph:')
print('━' * 70)
print(f'  {"CELEX":<15}  {"Document":<25}  {"Article":<10}  Context in EU AI Act')
print('  ' + '-' * 75)
for celex, doc, art_num, context in DANGLING_REFERENCES:
    print(f'  {celex:<15}  {doc:<25}  Art {art_num:<6}  {context}')

print()
print(f'Total dangling references to resolve: {len(DANGLING_REFERENCES)}')
print()
print('Strategy: for each, call resolve_article_from_eurlex() to get the text.')
print('Cache the results. In our knowledge graph, update the stub node: in_corpus=True.')

In [ ]:
# Resolve all dangling references
# (runs live against EUR-Lex when network is available; shows expected output otherwise)

print('Resolving all dangling references from EU AI Act...')
print('━' * 60)

resolved_stubs = {}

for celex, doc_name, art_num, context in DANGLING_REFERENCES:
    print(f'\n{doc_name} Article {art_num} (CELEX: {celex})')
    print(f'  Context: {context}')

    result = resolve_article_from_eurlex(celex, art_num)

    key = f'{celex}::Art::{art_num}'
    resolved_stubs[key] = {
        'node_id':    key,
        'doc_id':     celex,
        'doc_name':   doc_name,
        'article_num': art_num,
        'text':       result['text'],
        'in_corpus':  result['source'] not in ('error', 'network_blocked', 'not_found'),
        'source':     result['source'],
        'cellar_uri': result.get('cellar_uri'),
    }

    status = '✓ resolved' if resolved_stubs[key]['in_corpus'] else '✗ unresolved'
    print(f'  {status}  (source: {result["source"]})')

print()
print('━' * 60)
resolved_count = sum(1 for s in resolved_stubs.values() if s['in_corpus'])
print(f'Summary: {resolved_count}/{len(DANGLING_REFERENCES)} stubs resolved')

# Save the resolved stubs
import json
from pathlib import Path
with open('resolved_stubs.json', 'w') as f:
    json.dump(resolved_stubs, f, indent=2)
print('Saved: resolved_stubs.json')

In [ ]:
# How to integrate resolved stubs into the knowledge graph
# (Using the Python dict-based graph from your project — no Neo4j needed)

import json
from pathlib import Path
from collections import defaultdict

# Load existing data
DATA_DIR = Path('.')

with open(DATA_DIR / 'article_recital_map_llm.json') as f:
    arm = {int(k): v for k, v in json.load(f).items()}

# Simulate the knowledge graph as a dict (before adding Neo4j)
graph = {
    'nodes': {},   # node_id → node properties
    'edges': []    # list of (source_id, edge_type, target_id, properties)
}

# Add GDPR Article 9 as a resolved stub node
gdpr_art9_node = {
    'node_id':    'GDPR::Art::9',
    'doc_id':     '32016R0679',
    'doc_name':   'GDPR',
    'node_type':  'Article',
    'article_num': 9,
    'title':      'Processing of special categories of personal data',
    'text':       resolved_stubs.get('32016R0679::Art::9', {}).get('text', '(not yet fetched)'),
    'in_corpus':  resolved_stubs.get('32016R0679::Art::9', {}).get('in_corpus', False),
    'source':     'eurlex_live_fetch',
    'cellar_uri': resolved_stubs.get('32016R0679::Art::9', {}).get('cellar_uri'),
}
graph['nodes']['GDPR::Art::9'] = gdpr_art9_node

# Add EU AI Act Article 10 stub (it exists in our corpus)
euai_art10_node = {
    'node_id':    'EUAI::Art::10',
    'doc_id':     '32024R1689',
    'doc_name':   'EU AI Act',
    'node_type':  'Article',
    'article_num': 10,
    'title':      'Data and data governance',
    'in_corpus':  True,
}
graph['nodes']['EUAI::Art::10'] = euai_art10_node

# Add the cross-document REFERENCES edge
graph['edges'].append((
    'EUAI::Art::10',
    'REFERENCES',
    'GDPR::Art::9',
    {
        'ref_text': 'Regulation (EU) 2016/679',
        'context': 'special categories of personal data bias detection safeguards',
        'in_corpus': gdpr_art9_node['in_corpus'],
        'resolved': gdpr_art9_node['in_corpus'],
    }
))

print('Graph state after adding resolved GDPR stub:')
print(f'  Nodes: {len(graph["nodes"])}')
print(f'  Edges: {len(graph["edges"])}')
print()
print('GDPR Art 9 node:')
for k, v in gdpr_art9_node.items():
    if k != 'text':
        print(f'  {k}: {v}')
print(f'  text: {str(gdpr_art9_node["text"])[:100]}...' if gdpr_art9_node['text'] else '  text: (not fetched)')
print()
print('Edge:')
src, rel, tgt, props = graph['edges'][0]
print(f'  ({src}) -[{rel}]-> ({tgt})')
for k, v in props.items():
    print(f'    {k}: {v}')

---
## 9. SPARQL vs Cypher — Same Data, Different Languages

Both SPARQL and Cypher are graph query languages. Understanding them side by side
is the clearest way to see their differences. We will write the same logical queries
in both languages.

**Scenario**: We have a knowledge graph of the EU AI Act with:
- Article nodes connected to their Recitals by PROVIDES_RATIONALE_FOR edges
- Article nodes connected to referenced Articles by REFERENCES edges
- Obligation nodes connected to Articles by REQUIRES edges

**SPARQL uses triples** — every relationship is a triple pattern  
**Cypher uses patterns** — relationships are described as `(node)-[edge]->(node)` chains

In [ ]:
# Side-by-side comparison of SPARQL and Cypher
# For the same logical queries

comparisons = [
    {
        'task': 'Find all recitals that explain Article 5',
        'sparql': '''
PREFIX euai: <http://euai.example.org/ontology#>

SELECT ?recital_num ?reason
WHERE {
    ?recital   euai:recital_num          ?recital_num  .
    ?recital   euai:provides_rationale   ?article      .
    ?article   euai:article_num          5             .
    OPTIONAL {
        ?recital  euai:rationale_reason  ?reason  .
    }
}
ORDER BY ?recital_num
''',
        'cypher': '''
MATCH (r:Recital)-[e:PROVIDES_RATIONALE_FOR]->(a:Article {num: 5})
RETURN r.num AS recital_num, e.reason AS reason
ORDER BY r.num
'''
    },
    {
        'task': 'Find all articles that reference Article 6 (one-hop)',
        'sparql': '''
SELECT ?source_num ?source_title
WHERE {
    ?source   euai:references   ?target          .
    ?target   euai:article_num  6                .
    ?source   euai:article_num  ?source_num      .
    ?source   euai:title        ?source_title    .
}
ORDER BY ?source_num
''',
        'cypher': '''
MATCH (src:Article)-[:REFERENCES]->(tgt:Article {num: 6})
RETURN src.num AS source_num, src.title AS source_title
ORDER BY src.num
'''
    },
    {
        'task': 'Multi-hop: find obligations that flow from Art 16 via any chain',
        'sparql': '''
# SPARQL does not natively support variable-length paths.
# Property Paths (SPARQL 1.1) handle fixed-length alternatives:

SELECT ?obligation_name
WHERE {
    ?art16   euai:article_num   16   .
    ?art16   (euai:requires | euai:references/euai:requires)  ?obligation  .
    ?obligation  euai:obligation_name  ?obligation_name  .
}
''',
        'cypher': '''
# Cypher handles variable-length paths natively with *
# (Article 16) → any number of hops → (Obligation)

MATCH (a:Article {num: 16})-[:REQUIRES|REFERENCES*1..3]->(o:Obligation)
RETURN DISTINCT o.name AS obligation_name
'''
    },
    {
        'task': 'Cross-document: find GDPR articles referenced by EU AI Act',
        'sparql': '''
# RDF native — both graphs coexist in the same triple store
# No special handling needed for cross-document queries

SELECT ?euai_art_num ?gdpr_art_num ?context
WHERE {
    ?euai_art   euai:document   "EU_2024_1689"   .
    ?euai_art   euai:references  ?gdpr_art        .
    ?gdpr_art   euai:document   "EU_2016_679"    .
    ?euai_art   euai:article_num ?euai_art_num   .
    ?gdpr_art   euai:article_num ?gdpr_art_num   .
    OPTIONAL { ?euai_art  euai:ref_context  ?context  . }
}
''',
        'cypher': '''
# Cypher: use node labels or doc_id property to distinguish documents

MATCH (euai:Article:EUAIAct)-[r:REFERENCES]->(gdpr:Article:GDPR)
RETURN euai.num AS euai_art,
       gdpr.num AS gdpr_art,
       r.context AS context
'''
    },
]

for cmp in comparisons:
    print('='*70)
    print(f'TASK: {cmp["task"]}')
    print()
    print('SPARQL:')
    for line in cmp['sparql'].strip().split('\n'):
        print(f'  {line}')
    print()
    print('Cypher:')
    for line in cmp['cypher'].strip().split('\n'):
        print(f'  {line}')
    print()

In [ ]:
# Key differences summary — with implications for our project

differences = [
    (
        'Variable-length paths',
        'SPARQL 1.1 Property Paths: supports + (one or more) and * (zero or more)\n'
        '  but limited — no arbitrary traversal logic',
        'Cypher: MATCH (a)-[:REL*1..5]->(b) — variable depth, multiple rel types\n'
        '  with MATCH (a)-[:REL1|REL2*]->(b) — powerful graph traversal'
    ),
    (
        'Properties on edges',
        'RDF triples have no properties — you need reification (extra nodes)\n'
        '  e.g. to add reason to EXPLAINS: create an auxiliary node',
        'Cypher: -[e:PROVIDES_RATIONALE_FOR {reason: "...", confidence: 0.95}]->\n'
        '  edge properties are first-class — directly queryable'
    ),
    (
        'Cross-graph federation',
        'SPARQL SERVICE keyword: query multiple endpoints in ONE query\n'
        '  SERVICE <https://gdpr.example.org/sparql> { ... }\n'
        '  This is SPARQLs killer feature — no equivalent in Cypher',
        'Cypher: no built-in federation\n'
        '  Must handle cross-database queries in application code'
    ),
    (
        'For our EU AI Act project',
        'Would need to express reason/confidence as auxiliary triple nodes\n'
        '  More verbose, harder to query and update',
        'article_recital_reasons_llm.json maps DIRECTLY to edge properties\n'
        '  {reason, confidence, method} live on the edge — clean and queryable'
    ),
    (
        'When SPARQL wins',
        'EUR-Lex, national legal portals, parliamentary databases all expose\n'
        '  SPARQL endpoints. For cross-institution multi-corpus legal RAG,\n'
        '  SPARQL federation is the right tool.',
        'When you own your data and need fast application-layer queries,\n'
        '  Cypher/Neo4j is faster to develop, richer data model, better tooling.'
    ),
]

print('Key differences: SPARQL vs Cypher')
print('='*70)
print()
for topic, sparql_note, cypher_note in differences:
    print(f'━ {topic}')
    print(f'  SPARQL  : {sparql_note}')
    print(f'  Cypher  : {cypher_note}')
    print()

---
## 10. Limits of SPARQL for Our Project — What It Cannot Do Well

In [ ]:
# What EUR-Lex SPARQL CANNOT do for our project
# (Being explicit about limitations to avoid wasting time)

limitations = [
    {
        'limit': 'Cannot retrieve article text directly by article number',
        'detail': 'The Cellar SPARQL endpoint stores DOCUMENT metadata, not article-level text.\n'
                  'There is no triple like: (EU_AI_Act, hasArticle, Article5) with Article5 text.\n'
                  'You must fetch the full HTML and parse it yourself.',
        'workaround': 'Use SPARQL to get Cellar URI, then REST call, then parse HTML/XML.'
    },
    {
        'limit': 'The recital→article relationship is NOT in Cellar',
        'detail': 'Cellar does not know that Recital 40 explains Article 5.\n'
                  'This semantic legal knowledge is NOT part of the CDM ontology.\n'
                  'The 363-edge recital→article map we built with LLM is UNIQUE to our project.',
        'workaround': 'Keep our LLM-built map. SPARQL cannot replicate this knowledge.'
    },
    {
        'limit': 'Obligation/penalty nodes do not exist in Cellar',
        'detail': 'CDM stores regulatory metadata, not substantive legal content.\n'
                  'There is no triple for "Article 99 imposes a 30M EUR fine for providers."\n'
                  'All of that granular legal knowledge must be extracted by us.',
        'workaround': 'Extract obligations/penalties via LLM from our chunked text.'
    },
    {
        'limit': 'Rate limits and occasional downtime',
        'detail': 'The public SPARQL endpoint is shared infrastructure.\n'
                  'It has rate limits and occasional maintenance windows.\n'
                  'Do not rely on it for real-time responses in a production system.',
        'workaround': 'Cache all results. Pre-fetch known stubs at index time, not query time.'
    },
    {
        'limit': 'EU AI Act article content returns as unstructured HTML',
        'detail': 'Even after parsing, the HTML from EUR-Lex is not structured by paragraph number.\n'
                  'Extracting specifically "Article 5, paragraph 1, point (d)" is fragile.\n'
                  'The Formex XML (EUR-Lex\'s native format) is better structured but complex.',
        'workaround': 'Use EUR-Lex for bulk text discovery. Use our parsed chunks for fine-grained retrieval.'
    },
]

print('Limitations of EUR-Lex SPARQL for this project')
print('='*60)
print()
for i, lim in enumerate(limitations, 1):
    print(f'{i}. {lim["limit"]}')
    print(f'   Detail     : {lim["detail"]}')
    print(f'   Workaround : {lim["workaround"]}')
    print()

---
## Summary — When to Actually Use EUR-Lex SPARQL

In [ ]:
print('''
WHAT EUR-LEX SPARQL IS GOOD FOR IN OUR PROJECT
═══════════════════════════════════════════════

✓  Discover the CELEX number of any referenced regulation
     Example: EU AI Act says "Regulation (EU) 2016/679"
     SPARQL confirms: CELEX = 32016R0679, still in force, published 2016-05-04

✓  Get the Cellar URI to then fetch full regulation text
     SPARQL → URI → REST call → HTML → parse → article text

✓  Find what regulations AMEND the EU AI Act (forward citations)
     Important: Digital Omnibus 2025 proposes amendments to 2024/1689
     SPARQL automatically finds it because it cites 32024R1689

✓  Verify a regulation is still in force before using its content
     cdm:resource_legal_in-force returns true/false

✓  Discover language versions available for a regulation
     24 official EU language expressions accessible

✓  Find EuroVoc concept tags (thematic classification)
     Useful for topical clustering across multi-document corpus


WHAT EUR-LEX SPARQL CANNOT DO (for our project)
═══════════════════════════════════════════════════

✗  Return article text directly by article number
   (need full HTML fetch + parsing pipeline)

✗  Store our recital→article rationale map
   (this knowledge does not exist in CDM — we built it ourselves)

✗  Return obligations, penalties, or specific legal obligations
   (CDM is document metadata, not substantive legal content)

✗  Replace real-time retrieval in production
   (shared endpoint, rate limited, occasional downtime)


THE CORRECT MENTAL MODEL:
═════════════════════════

EUR-Lex SPARQL is a DISCOVERY and VERIFICATION tool,
not a retrieval tool for legal content.

Use it ONCE at index time to:
  1. Verify that a referenced regulation exists in EUR-Lex
  2. Get its Cellar URI for content fetching
  3. Pre-fetch and cache its article text as stub nodes
  4. Find forward citations to stay current with amendments

Then serve everything from your local knowledge graph at query time.
''')